# CyberSentinel-EU Stage 3 Part 1: Build the Vector Store
**Fatema Husain Hasan (202508958) MSc AI Thesis**

Embeds all 3,202 GDPR Enforcement Tracker records into ChromaDB.
One record = one chunk (summaries are single short paragraphs).

**Run:** paste Azure key in Cell 2, then Runtime -> Run all. (~8-10 min)

## 1. Setup

In [3]:
!pip install openai chromadb --quiet
import pandas as pd
import numpy as np
import json
import os
import time
from openai import AzureOpenAI
import chromadb
print('ready')

ready


## 2. Azure credentials and connection

In [4]:
AZURE_ENDPOINT = "https://cybersentinel-resource.openai.azure.com/"
AZURE_KEY = "PASTE_YOUR_KEY"
EMBED_DEPLOYMENT = "text-embedding-3-small"
API_VERSION = "2024-10-21"

client = AzureOpenAI(azure_endpoint=AZURE_ENDPOINT, api_key=AZURE_KEY, api_version=API_VERSION)
print("connection OK, vector length:",
      len(client.embeddings.create(model=EMBED_DEPLOYMENT, input="test").data[0].embedding))

connection OK, vector length: 1536


## 3. Load the full corpus (3,202 records)

In [5]:
REPO = "https://raw.githubusercontent.com/Fatimaxx24/202508958_IT9099_Thesis/main/"
df = pd.read_csv(REPO + "gdpr_enforcement_tracker_full.csv", low_memory=False)
print('corpus:', df.shape)

# clean: drop rows with no usable summary
df['Summary'] = df['Summary'].astype(str)
df = df[df['Summary'].str.strip().str.len() > 20].reset_index(drop=True)
print('records with usable summary:', len(df))
df[['ETid','Country','Sector','Date of Decision','Fine (EUR)','Summary']].head(5)

corpus: (3202, 12)
records with usable summary: 3201


,ETid,Country,Sector,Date of Decision,Fine (EUR),Summary
0,1,Austria,Industry and Commerce,12/9/2018,4800.0,Video surveillance was not sufficiently marked...
1,2,Austria,Accomodation and Hospitality,2018,1800.0,CCTV was unlawfully used. Sufficient informati...
2,3,Austria,Individuals and Private Associations,9/27/2018,300.0,A Dashcam was unlawfully used.
3,4,Austria,Individuals and Private Associations,12/20/2018,2200.0,The fine was imposed against a private person ...
4,5,Belgium,Public Sector and Education,5/28/2019,2000.0,The administrative fine was imposed for the mi...


## 4. Embed all records (batched + cached)
Batches of 50 for speed. Cache means re-running is free.

In [8]:
CACHE = "rag_embeddings_cache.json"
cache = json.load(open(CACHE)) if os.path.exists(CACHE) else {}
print('cached already:', len(cache))

texts = df['Summary'].tolist()
ids   = df['ETid'].astype(str).tolist()

def embed_batch(batch):
    r = client.embeddings.create(model=EMBED_DEPLOYMENT, input=batch)
    return [d.embedding for d in r.data]

vectors, B = [], 50
for i in range(0, len(texts), B):
    chunk_ids  = ids[i:i+B]
    chunk_txts = texts[i:i+B]
    todo_idx  = [j for j,cid in enumerate(chunk_ids) if cid not in cache]
    if todo_idx:
        got = embed_batch([chunk_txts[j][:8000] for j in todo_idx])
        for j,v in zip(todo_idx, got):
            cache[chunk_ids[j]] = v
    vectors.extend([cache[cid] for cid in chunk_ids])
    if (i//B) % 10 == 0:
        print(f'{min(i+B,len(texts))}/{len(texts)} embedded')
        json.dump(cache, open(CACHE,'w'))
json.dump(cache, open(CACHE,'w'))
emb = np.array(vectors)
print('DONE. embeddings shape:', emb.shape)

cached already: 3201
50/3201 embedded
550/3201 embedded
1050/3201 embedded
1550/3201 embedded
2050/3201 embedded
2550/3201 embedded
3050/3201 embedded
DONE. embeddings shape: (3201, 1536)


## 5. Build the ChromaDB collection
Each record stored with its vector + metadata (country, sector, year, fine, ETid)
so retrieval can cite sources and support filtering.

In [9]:
chroma = chromadb.PersistentClient(path="./chroma_gdpr")
try:
    chroma.delete_collection("gdpr_enforcement")
except Exception:
    pass
col = chroma.create_collection(name="gdpr_enforcement", metadata={"hnsw:space":"cosine"})

years = pd.to_datetime(df['Date of Decision'], errors='coerce', format='mixed').dt.year
metas = []
for i, r in df.iterrows():
    metas.append({
        "ETid": str(r['ETid']),
        "country": str(r['Country']),
        "sector": str(r['Sector']),
        "year": int(years[i]) if pd.notna(years[i]) else 0,
        "fine_eur": float(r['Fine (EUR)']) if pd.notna(r['Fine (EUR)']) else -1.0,
        "violation_type": str(r.get('Type of Violation',''))[:200],
        "source_url": str(r.get('Case Page',''))[:300],
    })

B = 500
for i in range(0, len(df), B):
    col.add(
        ids=ids[i:i+B],
        embeddings=[v.tolist() for v in emb[i:i+B]],
        documents=texts[i:i+B],
        metadatas=metas[i:i+B],
    )
    print(f'added {min(i+B,len(df))}/{len(df)}')
print('\nChromaDB collection size:', col.count())

added 500/3201
added 1000/3201
added 1500/3201
added 2000/3201
added 2500/3201
added 3000/3201
added 3201/3201

ChromaDB collection size: 3201


## 6. Smoke test - does retrieval work?

In [10]:
def search(query, k=5):
    qv = client.embeddings.create(model=EMBED_DEPLOYMENT, input=query).data[0].embedding
    res = col.query(query_embeddings=[qv], n_results=k)
    out = []
    for doc, meta, dist in zip(res['documents'][0], res['metadatas'][0], res['distances'][0]):
        out.append({'ETid': meta['ETid'], 'country': meta['country'],
                    'sector': meta['sector'], 'year': meta['year'],
                    'similarity': round(1-dist, 3), 'text': doc[:220]})
    return pd.DataFrame(out)

print("QUERY: ransomware attack on a hospital\n")
display(search("ransomware attack on a hospital"))
print("\nQUERY: employee sent personal data to the wrong email address\n")
display(search("employee sent personal data to the wrong email address"))

QUERY: ransomware attack on a hospital



,ETid,country,sector,year,similarity,text
0,2521,Belgium,Health Care,2024,0.579,"The Belgian DPA has fined a hospital EUR 200,0..."
1,2080,Italy,Health Care,2023,0.551,The Italian DPA has fined Asl Napoli 3 Sud EUR...
2,2468,Poland,Health Care,2024,0.513,"The Polish DPA has imposed a fine of EUR 9,200..."
3,3200,Ireland,Health Care,2026,0.507,"The Irish DPA has imposed a fine of EUR 300,00..."
4,1666,Ireland,Health Care,2023,0.498,"The Irish DPA has imposed a fine of EUR 460,00..."



QUERY: employee sent personal data to the wrong email address



,ETid,country,sector,year,similarity,text
0,171,Spain,"Media, Telecoms and Broadcasting",2020,0.732,The company had sent a contract with personal ...
1,219,Spain,Employment,2020,0.685,The company had sent the payroll of an employe...
2,263,Romania,"Media, Telecoms and Broadcasting",2020,0.641,The company has sent an email to a customer wh...
3,262,Romania,Transportation and Energy,2020,0.633,The company has sent an email to a client whic...
4,251,Hungary,Public Sector and Education,2019,0.622,The employee of the Directorate sent by mistak...


## 7. Save the vector store for reuse
Zip it so you can re-upload instead of re-embedding next session.

In [13]:
!zip -qr chroma_gdpr.zip chroma_gdpr
print('saved chroma_gdpr.zip - download it from the file browser (left sidebar)')
print('also download rag_embeddings_cache.json to avoid re-paying for embeddings')


zip error: Nothing to do! (try: zip -qr chroma_gdpr.zip . -i chroma_gdpr)
saved chroma_gdpr.zip - download it from the file browser (left sidebar)
also download rag_embeddings_cache.json to avoid re-paying for embeddings


## Next (Part 2)
- LangChain RetrievalQA with gpt-5-mini + citation/refusal prompt
- BM25 baseline over the same corpus
- The frozen 15-question benchmark and four-dimension scoring